<a href="https://colab.research.google.com/github/AlmasMalik66/DataScience-AI-Assignments/blob/main/Week11/Assignment11_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Week 11 – Natural Language Processing (NLP)**

**Project Title: Fake News Detection**

This week focuses on **Natural Language Processing (NLP)** techniques used to transform raw news text into meaningful numerical representations for machine learning and deep learning models.

Since **Fake News Detection is a text-based project**, NLP is a core requirement. Therefore, all NLP techniques were applied directly to the project dataset, unlike tabular projects where NLP is demonstrated separately.

**Dataset Preparation (Project-Based)**

The cleaned file did not contain labels, so labels were created manually using the original datasets.

In [1]:
import pandas as pd

df_fake = pd.read_csv("Fake.csv", on_bad_lines='skip')
df_true = pd.read_csv("True.csv", on_bad_lines='skip')

df_fake['label'] = 1   # Fake news
df_true['label'] = 0   # Real news

df_full = pd.concat([df_fake, df_true], ignore_index=True)

df_full['combined_text'] = df_full['title'] + " " + df_full['text']
df_full = df_full.dropna(subset=['combined_text'])

df_full.head()


,title,text,subject,date,label,combined_text
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1,Donald Trump Sends Out Embarrassing New Year’...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1,Drunk Bragging Trump Staffer Started Russian ...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1,Sheriff David Clarke Becomes An Internet Joke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1,Trump Is So Obsessed He Even Has Obama’s Name...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1,Pope Francis Just Called Out Donald Trump Dur...


**Class Task – Tokenization & Text Processing (Project Dataset)**

The class task focused on understanding how raw text is converted into structured data using NLP preprocessing techniques.

**Step 1: Import NLP Libraries**

In [2]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

**Step 2: Text Cleaning (Lowercasing & Symbol Removal)**

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df_full['clean_text'] = df_full['combined_text'].apply(clean_text)
df_full[['combined_text', 'clean_text']].head()


,combined_text,clean_text
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump sends out embarrassing new years...
1,Drunk Bragging Trump Staffer Started Russian ...,drunk bragging trump staffer started russian ...
2,Sheriff David Clarke Becomes An Internet Joke...,sheriff david clarke becomes an internet joke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,trump is so obsessed he even has obamas name ...
4,Pope Francis Just Called Out Donald Trump Dur...,pope francis just called out donald trump dur...


**Step 3: Tokenization, Stopword Removal & Lemmatization**

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

# Apply preprocessing
df_full['processed_text'] = df_full['clean_text'].apply(preprocess)
df_full[['clean_text', 'processed_text','combined_text', 'label']].head()


,clean_text,processed_text,combined_text,label
0,donald trump sends out embarrassing new years...,donald trump sends embarrassing new year eve m...,Donald Trump Sends Out Embarrassing New Year’...,1
1,drunk bragging trump staffer started russian ...,drunk bragging trump staffer started russian c...,Drunk Bragging Trump Staffer Started Russian ...,1
2,sheriff david clarke becomes an internet joke...,sheriff david clarke becomes internet joke thr...,Sheriff David Clarke Becomes An Internet Joke...,1
3,trump is so obsessed he even has obamas name ...,trump obsessed even obamas name coded website ...,Trump Is So Obsessed He Even Has Obama’s Name...,1
4,pope francis just called out donald trump dur...,pope francis called donald trump christmas spe...,Pope Francis Just Called Out Donald Trump Dur...,1


# **Assignment 11 – NLP Feature Engineering (TF-IDF)**

After preprocessing, text was converted into numerical features using TF-IDF, which is suitable for fake news classification.

**Step 4: TF-IDF Vectorization**

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df_full['processed_text']).toarray()
y = df_full['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


**Step 5: Train a Baseline Classifier**

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("TF-IDF Model Accuracy:", accuracy_score(y_test, y_pred))


TF-IDF Model Accuracy: 0.9884187082405346
